In [ ]:
import os
from pathlib import Path
from IPython.display import display
import pysacsnow
from pysacsnow import Controller
from pysacsnow import SACSMA
import pandas as pd

## Running SAC-SMA Model for 1 Hawaii basin
In this example, we are going to use the example data and input files to execute the SAC-SMA model.

### Controller

In [ ]:
# Update this to work on your machine

# If you opened Jupyter in the directory containing this file, you can run the following command to get the path
# EXAMPLES_DIR = Path.cwd().parent

# Define explicit path to examples directory
study_dir = Path("/home/kvanwerkhoven/projects/pysac/hawaii")

# Define and load the configuration/control file
location_id = '16103000'
yaml_file = Path(study_dir, "configurations", "sac", f"sac.control.{location_id}.yaml")
control = Controller.load_from_yaml(yaml_file)
control

# Print the control file config
control.config

In [ ]:
control.config.simulation.start_date

This generated the model Controller. The model controller "controls" (see what we did there?) and manages the the model simulation in time. The primary purpose of the Controller is to keep track of and advance the model simulation in time. However the Controller class also loads the simulation "config" which is a ConfigFile object (discussed in notebook 01).

### Initializing the model
Now that the data paths are all updated, we can instantiate the model. In this example, we're going to use **numpy** for the `calc_method`. The other available `calc_method` is **numba**. In the numba method, the code uses [numba njit](https://numba.pydata.org/) to parallelize the code across the spatial dimensions. This is typically faster once the number of `hrus` (hydrologic response units, i.e., spatial model units/subbasins) reaches a large enough number (> 5). This is because the code compiles with running with numba so sometimes the overhead of compiling first does not beat the numpy simulation which runs in series. 

The SACSMA class is an [base class](https://docs.python.org/3/library/abc.html) of the `Process` class. So we will refer to this later as a `Process`. By doing this we can make every model (i.e., sac, snow, unit hg, etc.) class inherit methods from `Process` class, that way they all use the same methods and functions so that there is consistency. TLDR; once you learn to run the SACSMA class, running SNOW17 is indentical.

In [ ]:
sacsma = SACSMA(control=control, calc_method="numpy")
sacsma

In [ ]:
# Print the model dimensions; in this case, the current model has dimensions are always hrus, i.e.,
# spatial model units. If another model used reaches or something else, it would be listed here.
sacsma.get_dimensions()

In [ ]:
# Print the model dimension sizes
sacsma.nhru

In [ ]:
# Print the model input names. These are the names of the input data that the model needs to run.
# Note that the model will look for these input data if they are not present it will just a nan value.
# In some cases the nan will through an error, if the model cannot run with missing data. For example, 
# the model needs precipitation data, and pet to run, but the aesc input is optional.
sacsma.get_inputs()

### Running the model
The model executes for all dimensions (i.e., nhrus) for a single time step. We can run and stop the model at any point, and similarly update/change any of the model variables. Obviously, for most standard simulations, the user would just set up the model to run through all time-steps with out any changes. 

However, if one was doing data assimilation updates, or calibration, it could be done while the model is initialized in memory and without stopping, writing out files, or making modifications to files, and then restarting the model.

The model is executed in 4 steps:
1. Advance the `Controller`. Advancing the controller moves the time step along.
2. Advance the `SACSMA` process. This advances the model inputs and variables forward. I.e., advancing the inputs results in the model getting the appropriate forcing data for the current period. Advancing variables, is dependant on the model process but could include keeping track of or modifing some state variable between model calulations.
3. Calculate the `SACSMA` process. This executes the model process algorithms for a single time step (in this example SAC-SMA).
4. Output the `SACSMA` process results. This is optional, but this outputs the process terms for a time step. This can very on how it was setup, but this could include writing out to a file, or other in-memory variable. To reduce IO and speed up simulation times, we do not output the results to a file every timestep (or every time .output() is called). Instead we write the outputs to in an in-memory "states buffer". There is a default "MAX_BUFFER_SIZE" which will write the data to a file if the buffer reaches max size (so that you don't run into memory issues) and/or when `sacsma.finalize()` is called.

Lastly, `sacsma.finalize()` can be used to finalize and output Process terms as necessary. This is typically used at the end of simulation.

In [ ]:
# Current path to the output directory
print(control.config.simulation.output_path)
# Update with new full path

In [ ]:
control.config.simulation.output_path = Path('/home/kvanwerkhoven/projects/pysac/hawaii/data/output/output_16103000.csv')

In [ ]:
sacsma.reset()
control.current_time

In [ ]:
# NOTE: Since we already ran the model for 50 time steps, you would need to 
# subtract 50 from the control.n_times if you wanted to run the model for the full 
# simulation length at this point.

for tt in range(control.n_times-40):
    control.advance()
    sacsma.advance()
    sacsma.calculate()
    sacsma.output()
# Save the model outputs
sacsma.finalize()


In [ ]:
# Read in results from the output file
output_df = pd.read_csv(control.config.simulation.output_path)
out_tci = output_df[output_df['variable_name'] == 'tci'].copy()

In [ ]:
import holoviews as hv
hv.extension('bokeh') 
from bokeh.models import DatetimeTickFormatter, MonthsTicker

In [ ]:
# Create the curve
out_tci['value_time'] = pd.to_datetime(out_tci['value_time'])

curve = hv.Curve(out_tci, kdims='value_time', vdims='value', label='sim').opts(
    width=900,
    height=400,
    xlabel='Datetime',
    ylabel='TCI',
    tools=['hover'],
    xrotation=90,    
)

obs = Path(study_dir, 'data','observations', f'QIN_{location_id}.csv')
df_obs = pd.read_csv(obs)

df_obs['time'] = pd.to_datetime(df_obs['time'])
curve_obs = hv.Curve(df_obs, kdims='time', vdims='QIN', label='obs').opts(
    width=900,
    height=400,
    xlabel='Datetime',
    tools=['hover'],
    xrotation=90,    
)

curve * curve_obs

In [ ]:
pd.read_csv('/home/kvanwerkhoven/projects/pysac/hawaii/data/forcing/forcing.sac.16103000.csv')

In [ ]:
# Save the model outputs to disk (if needed)

sacsma.finalize()

Note: this is slow since we are reading in a model that is saving ALL of the state variables at each time steps; to speed this up 
and only save select variables, you would need to modify the config file with output_variables: - name: minimal
See sac.control.ANBC2.yaml for an example

